In [ ]:
import zipfile
import json
from tqdm.auto import tqdm
import os
import requests
import pandas as pd

In [ ]:
projects = ["epsd2/admin/ur3"]

In [ ]:
def oracc_download(project_list):
    CHUNK = 1024

    for project in project_list:
        proj = project.replace("/", "-")
        url = f"https://oracc.museum.upenn.edu/json/{proj}.zip"
        localfile = f"jsonzip/{proj}.zip"

        try:
            with requests.get(url, stream=True, timeout=30) as r:

                r.raise_for_status() # catch any type of error in handling the request

                total_size = int(r.headers.get("content-length", 0))
                tqdm.write(f"Saving {url} → {localfile}")

                with (
                    open(localfile, "wb") as f,
                    tqdm(
                        total=total_size,
                        unit="B",
                        unit_scale=True,
                        desc=project,
                    ) as bar,
                ):
                    for chunk in r.iter_content(chunk_size=CHUNK):
                        if chunk:
                            f.write(chunk)
                            bar.update(len(chunk))

        except requests.exceptions.RequestException as e:
            tqdm.write(f"ERROR downloading {project}: {e}")

In [ ]:
oracc_download(projects)

In [ ]:
def parsejson(text, lemma_list):
    for JSONobject in text["cdl"]:
        if "cdl" in JSONobject: 
            parsejson(JSONobject, lemma_list)
        elif "f" in JSONobject:          # copy all the lemmatization data in the variable word
            word = JSONobject["f"]
            reference = JSONobject["ref"]
            if not word.get("lang", "").startswith("sux"): #only Sumerian and Emesal
                continue
            if word.get("pos", "") in ["PN", "SN", "GN"]: # no proper nouns or geographical names
                continue
            citation_form = word.get("cf", "")

            if citation_form.endswith("um") and len(citation_form) > 3:
                lemma_list.append([citation_form,  reference])
    return lemma_list

In [ ]:
file = "jsonzip/epsd2-admin-ur3.zip"
project = projects[0]
ids_ = []
lemma_list = []
z = zipfile.ZipFile(file)       # create a Zipfile object
files = z.namelist()     # list of all the files in the ZIP
files = [name for name in files if "corpusjson" in name and name.endswith('.json')]                                                                                                  #that holds all the P, Q, and X numbers.

for filename in tqdm(files, desc = project):      #iterate over the file names
    id_no = filename[-13:-5]
    id_text = project + id_no # id_text is, for instance, blms/P414332
    ids_.append(id_text)
    text = z.read(filename).decode("utf-8")
    data_json = json.loads(text)
    akk_words = parsejson(data_json, lemma_list)

In [ ]:
akk_words_df = pd.DataFrame(akk_words)
akk_words_df.columns = ["lemma", "reference"]

In [ ]:
akk_words2_df = akk_words_df.groupby("lemma").agg({'reference': ','.join})

In [ ]:
akk_words2_df